# DreamerV3 on Google Colab

This notebook trains [DreamerV3](https://github.com/danijar/dreamerv3) on a few example environments entirely inside Colab. It is tested against the free-tier GPU runtime.

**Runtime setup (important):**

1. Click `Runtime -> Change runtime type`.
2. Set `Hardware accelerator` to `GPU` (T4 is fine).
3. Run the cells below in order.

You can skip any environment section you don't want to try. The Minecraft section is optional and heavy — read its notes before running it.

## 1. Verify the runtime

In [ ]:
!nvidia-smi || echo 'No GPU detected — switch the runtime type to GPU.'
import sys
print('Python:', sys.version)

## 2. Install DreamerV3 and shared dependencies

This takes ~2 minutes. JAX is installed with CUDA 12 wheels so that the agent trains on the Colab GPU.

In [ ]:
%pip install -q --upgrade pip
%pip install -q 'dreamerv3>=3.0.0' 'gymnasium>=0.29' tensorboard ruamel.yaml cloudpickle
%pip install -q --upgrade 'jax[cuda12]' 'jaxlib'

import jax
print('JAX devices:', jax.devices())

## 3. Clone the examples

Pull this repository so the example scripts are available on the Colab filesystem.

In [ ]:
import os
if not os.path.isdir('/content/rl-dreamer'):
    !git clone --depth 1 https://github.com/kuds/rl-dreamer.git /content/rl-dreamer
%cd /content/rl-dreamer
!ls examples

## 4. (Optional) Persist logs to Google Drive

Colab wipes `/root/logdir` when the runtime resets, destroying checkpoints, replay buffers, videos, and dreams. Mounting Google Drive keeps them across sessions.

Set `USE_DRIVE = True` to mount Drive; leave it `False` for ephemeral storage. All downstream cells read `$LOGDIR_ROOT` so you only have to configure this once.

In [ ]:
USE_DRIVE = False  # set to True to persist logs under Google Drive

import os

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    LOGDIR_ROOT = '/content/drive/MyDrive/dreamerv3'
else:
    LOGDIR_ROOT = '/root/logdir'

os.makedirs(LOGDIR_ROOT, exist_ok=True)
os.environ['LOGDIR_ROOT'] = LOGDIR_ROOT
print('Logs will be written under:', LOGDIR_ROOT)

## 5. Launch TensorBoard (optional)

Run this cell to watch live training curves while the agent learns.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $LOGDIR_ROOT

## 6. CartPole (smoke test)

Trains in about a minute. Good check that the install actually works.

In [ ]:
!python examples/01_gym_cartpole.py \
    --logdir $LOGDIR_ROOT/cartpole \
    --run.steps 20000

## 7. DeepMind Control — Walker Walk (from pixels)

A classic continuous-control benchmark. Installs `dm_control` then trains for a short number of steps so the cell finishes on a free GPU. Increase `run.steps` for real results.

In [ ]:
%pip install -q dm_control
# Colab needs a software GL renderer for MuJoCo offscreen rendering.
import os
os.environ['MUJOCO_GL'] = 'egl'

In [ ]:
!python examples/02_dmc_walker.py \
    --logdir $LOGDIR_ROOT/dmc_walker \
    --run.steps 50000 \
    --batch_size 8

## 8. Atari — Pong

Installs the ALE ROMs via the Gymnasium `accept-rom-license` extra.

In [ ]:
%pip install -q 'gymnasium[atari,accept-rom-license]' ale-py

In [ ]:
!python examples/03_atari_pong.py \
    --logdir $LOGDIR_ROOT/atari_pong \
    --run.steps 100000 \
    --batch_size 8

## 9. Crafter

The 2D Minecraft-inspired benchmark — a good stand-in if you want Minecraft-like dynamics without the Java / MineRL setup.

In [ ]:
%pip install -q crafter
!python examples/04_crafter.py \
    --logdir $LOGDIR_ROOT/crafter \
    --run.steps 100000 \
    --batch_size 8

## 10. Minecraft via MineRL (optional, heavy)

DreamerV3's flagship Minecraft diamond task runs on top of [MineRL](https://minerl.readthedocs.io), which in turn launches a real Minecraft JVM.

**Caveats on Colab:**
- Requires Java 8 (installed below) and an `xvfb` display.
- The first episode takes several minutes as Minecraft boots.
- Training a diamond-collecting agent from scratch takes tens of millions of env steps — well beyond a single Colab session. Use this cell as a demonstration of the plumbing rather than expecting paper-quality results.

If the install is too slow or unstable for your Colab, skip straight to Crafter above.

In [ ]:
# Install Java 8 and xvfb for the MineRL backend.
!apt-get -qq update
!apt-get -qq install -y openjdk-8-jdk xvfb > /dev/null
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-8-openjdk-amd64'
!java -version

In [ ]:
%pip install -q git+https://github.com/minerllabs/minerl@v1.0.2

In [ ]:
# Launch the example under xvfb so MineRL has a display to render into.
!xvfb-run -a python examples/06_minecraft.py \
    --logdir $LOGDIR_ROOT/minecraft \
    --run.steps 20000 \
    --batch_size 4

## 11. Inspecting trained checkpoints

The `Evaluator` class from `scripts/evaluate.py` reloads a saved agent and runs deterministic rollouts. It returns a numpy array of returns you can summarize however you like.

In [ ]:
import os
from pathlib import Path

from scripts.evaluate import Evaluator

evaluator = Evaluator(
    task='gym_CartPole-v1',
    preset='size1m',
    logdir=Path(os.environ['LOGDIR_ROOT']) / 'cartpole',
)
returns = evaluator.run(episodes=5)

print(f'\nMean return: {returns.mean():.3f} +/- {returns.std():.3f} (n={len(returns)})')

## 12. Recording rollout videos

The `Recorder` class from `scripts/record.py` writes one MP4 per episode to `<logdir>/videos/` and returns a list of dicts with each episode's return and path. Install the `imageio` ffmpeg backend first, then construct the recorder with the same `task` / `preset` / `logdir` you used for training.

In [ ]:
%pip install -q imageio imageio-ffmpeg

import os
from pathlib import Path

from scripts.record import Recorder

rec = Recorder(
    task='gym_CartPole-v1',
    preset='size1m',
    logdir=Path(os.environ['LOGDIR_ROOT']) / 'cartpole',
)
episodes = rec.record(episodes=3, fps=30)

print(f'\nSaved {len(episodes)} video(s):')
for r in episodes:
    print(f"  return={r['return']:.2f}  frames={r['frames']}  ->  {r['path']}")

In [ ]:
import os
# Embed the first video inline so you can watch it without downloading.
import glob
from IPython.display import HTML
from base64 import b64encode

videos = sorted(glob.glob(f'{os.environ["LOGDIR_ROOT"]}/cartpole/videos/*.mp4'))
if not videos:
    print('No videos found - did the recording cell above run successfully?')
else:
    data = b64encode(open(videos[0], 'rb').read()).decode()
    HTML(f'<video width=480 controls src="data:video/mp4;base64,{data}"></video>')

## 13. Visualizing world-model dreams

This section uses the `DreamVisualizer` class from `scripts/visualize_dreams.py` as a library. It's the complement to `record.py`: instead of showing the agent acting in the *real* environment, it shows what the **world model predicts**.

Under the hood `DreamVisualizer.generate()` calls `agent.report()` on a batch of real observations and writes:

- **Reconstruction / open-loop dream MP4s** — the decoder's output when fed posterior latents. The first few frames are grounded by real observations; the rest are rolled out from the prior alone, so you can literally see the model predicting the next frames.
- **Contact-sheet PNGs** of the same frames for quick inspection.

Dreams only make sense for pixel-based tasks, so we run them on the Crafter checkpoint trained in section 9.

In [ ]:
import os
from pathlib import Path
from base64 import b64encode
from IPython.display import HTML

from scripts.visualize_dreams import DreamVisualizer

CRAFTER_LOGDIR = Path(os.environ['LOGDIR_ROOT']) / 'crafter'

viz = DreamVisualizer(
    task='crafter_reward',
    preset='size25m',
    logdir=CRAFTER_LOGDIR,
)
saved = viz.generate(fps=10)

print(f'{len(saved)} file(s) written; batch source: {viz.batch_source}')
for key, path, shape in saved:
    kind = 'video' if path.suffix == '.mp4' else 'image'
    print(f'  [{kind:5s}] {key:30s} {str(shape):20s} -> {path.name}')

In [ ]:
# Embed the first dream MP4 inline. Prefers the open-loop imagination video.
import glob, os
from base64 import b64encode
from IPython.display import HTML

dreams = sorted(glob.glob(f"{os.environ['LOGDIR_ROOT']}/crafter/dreams/*.mp4"))
if not dreams:
    print('No dream videos found - did the cell above run successfully?')
else:
    preferred = [p for p in dreams if 'openl' in p.lower()] or dreams
    data = b64encode(open(preferred[0], 'rb').read()).decode()
    HTML(f'<video width=480 controls src="data:video/mp4;base64,{data}"></video>')

## 14. Next steps

- Increase `--run.steps` and model preset (`--preset size25m` or `size50m`) for real results.
- Set `USE_DRIVE = True` in section 4 to persist checkpoints, videos, and dreams to Google Drive.
- Plug in your own environment using `examples/07_custom_env.py` as a template.
- Read [`docs/architecture.md`](../docs/architecture.md), [`docs/LEARNING.md`](../docs/LEARNING.md), and [`docs/tensorboard_guide.md`](../docs/tensorboard_guide.md) for background and metric explanations.